# Mobile Money Agent Network: Data Audit and Cleaning

Auditing, cleaning and summarising the transaction extract from the legacy system before any analysis is trusted.

## Setup

In [1]:
import os
import numpy as np
import pandas as pd

os.makedirs("outputs", exist_ok=True)

## Create the Data

This cell saves `generate_data.py`, which contains both dataset generators from this week unchanged, and the next cell runs it to create the raw CSVs.

In [2]:
%%writefile generate_data.py
import numpy as np
import pandas as pd

# Engagement 1: Duka Electronics (seed 2026)
rng = np.random.default_rng(2026)
n = 1000
products = {"Phone": 15000, "Laptop": 65000, "Headphones": 2500,
            "Tablet": 28000, "Charger": 1200}

sales = pd.DataFrame({
    "order_id": range(1001, 1001 + n),
    "order_date": pd.to_datetime("2026-01-01")
                  + pd.to_timedelta(rng.integers(0, 180, n), unit="D"),
    "branch": rng.choice(["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret"], n,
                         p=[0.4, 0.2, 0.15, 0.15, 0.1]),
    "product": rng.choice(list(products), n),
    "quantity": rng.integers(1, 6, n),
    "channel": rng.choice(["In-store", "Online", "Phone order"], n,
                          p=[0.55, 0.35, 0.10]),
})
sales["unit_price"] = sales["product"].map(products)
sales.to_csv("duka_sales.csv", index=False)

# Engagement 2: Mobile money agent network (seed 44)
rng = np.random.default_rng(44)
n = 600

agents = pd.DataFrame({
    "agent_id": [f"AG{i:03d}" for i in range(1, 21)],
    "county": rng.choice(["Nairobi", "Kiambu", "Nakuru", "Mombasa", "Kisumu"], 20),
    "onboarded": pd.date_range("2024-01-01", periods=20, freq="MS")
                   .strftime("%Y-%m-%d"),
})

txns = pd.DataFrame({
    "txn_id": [f"TX{i:05d}" for i in range(1, n + 1)],
    "agent_id": rng.choice(agents["agent_id"], n),
    "txn_type": rng.choice(["Deposit", "withdrawal", "DEPOSIT ",
                            "Withdrawal", "deposit"], n),
    "amount": rng.lognormal(7.5, 0.8, n).round(0).astype(str),
    "txn_date": (pd.to_datetime("2026-03-01")
                 + pd.to_timedelta(rng.integers(0, 60, n), unit="D"))
                .strftime("%Y-%m-%d"),
})

idx = rng.choice(n, 30, replace=False)
txns.loc[idx[:15], "amount"] = np.nan
txns.loc[idx[15:20], "amount"] = "N/A"
txns.loc[idx[20:25], "amount"] = "250000"
txns.loc[idx[25:], "txn_date"] = txns.loc[idx[25:], "txn_date"].str.replace("-", "/")
txns = pd.concat([txns, txns.sample(12, random_state=1)], ignore_index=True)
txns.loc[rng.choice(len(txns), 8, replace=False), "agent_id"] = "AG099"

agents.to_csv("agents.csv", index=False)
txns.to_csv("agent_transactions.csv", index=False)

print("Created duka_sales.csv, agents.csv and agent_transactions.csv")


Overwriting generate_data.py


In [3]:
%run generate_data.py

Created duka_sales.csv, agents.csv and agent_transactions.csv


## 1. Load the Data

The transactions are loaded twice. The raw text version keeps values such as "N/A" exactly as typed so the audit can count them. The normal version is what pandas gives by default, where "N/A" and blanks both become missing.

In [4]:
raw_text = pd.read_csv("agent_transactions.csv", dtype=str, keep_default_na=False)
txns = pd.read_csv("agent_transactions.csv")
agents = pd.read_csv("agents.csv")

print("Transactions:", txns.shape)
print("Agents:", agents.shape)
txns.head()

Transactions: (612, 5)
Agents: (20, 3)


,txn_id,agent_id,txn_type,amount,txn_date
0,TX00001,AG009,deposit,353.0,2026-03-03
1,TX00002,AG020,deposit,1238.0,2026-04-27
2,TX00003,AG008,DEPOSIT,9129.0,2026-04-29
3,TX00004,AG009,deposit,652.0,2026-04-24
4,TX00005,AG009,withdrawal,478.0,2026-04-17


## 2. Data Quality Audit

In [5]:
valid_types = ["Deposit", "Withdrawal"]

audit = pd.DataFrame([
    ["Duplicate transaction IDs", "Uniqueness", "txn_id",
     txns["txn_id"].duplicated().sum()],
    ["Inconsistent transaction type spelling or case", "Consistency", "txn_type",
     (~txns["txn_type"].isin(valid_types)).sum()],
    ["Blank amount", "Completeness", "amount",
     (raw_text["amount"] == "").sum()],
    ["Amount typed as N/A", "Validity", "amount",
     (raw_text["amount"] == "N/A").sum()],
    ["Repeated KES 250,000 amount", "Accuracy", "amount",
     (txns["amount"] == 250000).sum()],
    ["Date uses / instead of -", "Validity", "txn_date",
     txns["txn_date"].str.contains("/").sum()],
    ["Agent ID not in agent register", "Validity", "agent_id",
     (~txns["agent_id"].isin(agents["agent_id"])).sum()],
], columns=["issue", "dimension", "column", "rows_affected"])

audit

,issue,dimension,column,rows_affected
0,Duplicate transaction IDs,Uniqueness,txn_id,12
1,Inconsistent transaction type spelling or case,Consistency,txn_type,384
2,Blank amount,Completeness,amount,15
3,Amount typed as N/A,Validity,amount,7
4,"Repeated KES 250,000 amount",Accuracy,amount,5
5,Date uses / instead of -,Validity,txn_date,5
6,Agent ID not in agent register,Validity,agent_id,8


## 3. Cleaning Pipeline

The cleaning rules are saved in `cleaning_pipeline.py` so the same rules can be run on next week's extract. The cell below writes the module and the next cell imports and calls it.

In [6]:
%%writefile cleaning_pipeline.py
import pandas as pd


def clean_agent_data(txns, agents, suspect_amount=250000):
    """Clean the agent transactions extract.

    Removes duplicate transactions, standardises txn_type, converts amount
    and txn_date, imputes missing amounts with the median, flags IQR
    outliers and suspected entry errors, and merges in the agent county.

    Returns (clean_df, log).
    """
    log = []

    # Uniqueness: one row per transaction ID
    out = txns.drop_duplicates(subset="txn_id", keep="first").copy()
    log.append(f"Uniqueness: removed {len(txns) - len(out)} duplicate transactions")

    # Consistency: exactly two transaction types
    before = (~out["txn_type"].isin(["Deposit", "Withdrawal"])).sum()
    out["txn_type"] = out["txn_type"].str.strip().str.title()
    log.append(f"Consistency: standardised txn_type in {before} rows")

    # Validity: amount to numbers, N/A and blanks become missing
    out["amount"] = pd.to_numeric(out["amount"], errors="coerce")

    # Outliers: IQR method on the real amounts, before imputation
    q1, q3 = out["amount"].quantile([0.25, 0.75])
    upper = q3 + 1.5 * (q3 - q1)
    out["is_outlier"] = out["amount"] > upper
    out["suspected_entry_error"] = out["amount"] == suspect_amount
    log.append(f"Accuracy: flagged {out['is_outlier'].sum()} IQR outliers "
               f"above KES {upper:,.0f}")
    log.append(f"Accuracy: flagged {out['suspected_entry_error'].sum()} amounts of "
               f"KES {suspect_amount:,} as suspected entry errors")

    # Completeness: flag first, then fill with the median
    out["amount_imputed"] = out["amount"].isna()
    median = out.loc[~out["suspected_entry_error"], "amount"].median()
    out["amount"] = out["amount"].fillna(median)
    log.append(f"Completeness: imputed {out['amount_imputed'].sum()} missing or "
               f"invalid amounts with the median of KES {median:,.0f}")

    # Validity: one ISO date format
    bad_dates = out["txn_date"].astype(str).str.contains("/").sum()
    out["txn_date"] = pd.to_datetime(out["txn_date"].astype(str).str.replace("/", "-"))
    log.append(f"Validity: fixed {bad_dates} dates written with /")

    # Merge with the agent register
    out = out.merge(agents[["agent_id", "county"]], on="agent_id",
                    how="left", indicator=True)
    out["unknown_agent"] = out["_merge"] == "left_only"
    out["county"] = out["county"].fillna("Unknown")
    out = out.drop(columns="_merge")
    log.append(f"Validity: {out['unknown_agent'].sum()} transactions have agents "
               "missing from the register, kept with county Unknown")

    return out, log

Overwriting cleaning_pipeline.py


In [7]:
from cleaning_pipeline import clean_agent_data

clean, log = clean_agent_data(txns, agents)

print("Cleaning log")
for line in log:
    print("-", line)
print()
print("Rows before cleaning:", len(txns))
print("Rows after cleaning: ", len(clean))

Cleaning log
- Uniqueness: removed 12 duplicate transactions
- Consistency: standardised txn_type in 376 rows
- Accuracy: flagged 40 IQR outliers above KES 6,084
- Accuracy: flagged 5 amounts of KES 250,000 as suspected entry errors
- Completeness: imputed 20 missing or invalid amounts with the median of KES 1,770
- Validity: fixed 5 dates written with /
- Validity: 7 transactions have agents missing from the register, kept with county Unknown

Rows before cleaning: 612
Rows after cleaning:  600


## 4. Check Each Cleaning Step

In [8]:
print("Uniqueness")
print("Duplicate rows removed:", len(txns) - len(clean))
print("Duplicate txn_id left:", clean["txn_id"].duplicated().sum())
print()
print("TX00405 appears twice in the raw file with different agent IDs")
print(txns[txns["txn_id"] == "TX00405"])

Uniqueness
Duplicate rows removed: 12
Duplicate txn_id left: 0

TX00405 appears twice in the raw file with different agent IDs
      txn_id agent_id txn_type  amount    txn_date
404  TX00405    AG014  Deposit  1936.0  2026-04-03
601  TX00405    AG099  Deposit  1936.0  2026-04-03


In [9]:
print("Consistency")
print("Before:")
print(txns["txn_type"].value_counts())
print()
print("After:")
print(clean["txn_type"].value_counts())

Consistency
Before:
txn_type
deposit       142
withdrawal    131
Deposit       128
DEPOSIT       111
Withdrawal    100
Name: count, dtype: int64

After:
txn_type
Deposit       373
Withdrawal    227
Name: count, dtype: int64


In [10]:
print("Validity and types")
print(clean[["amount", "txn_date"]].dtypes)
print()
print("Date range:", clean["txn_date"].min().date(), "to", clean["txn_date"].max().date())
print()
print("Completeness")
print("Amounts imputed:", clean["amount_imputed"].sum())
print("Missing amounts left:", clean["amount"].isna().sum())

Validity and types
amount             float64
txn_date    datetime64[ns]
dtype: object

Date range: 2026-03-01 to 2026-04-29

Completeness
Amounts imputed: 20
Missing amounts left: 0


**How missing amounts were treated:** 20 amounts were blank or typed as "N/A". They were marked in the `amount_imputed` column first, and then filled with the median amount of KES 1,770. The median was used, not the mean, because mobile money amounts are skewed and the mean would be pulled up by a few large values. The five suspected KES 250,000 entries were left out when calculating it. The flag means anyone using the clean file can tell which amounts are estimates and leave them out if needed. Together the imputed values add KES 35,400 to the totals, about 2.5% of the clean value. The business should look up the real amounts in the agents' records.

## 5. Outliers

In [11]:
real_amounts = clean.loc[~clean["amount_imputed"], "amount"]
q1, q3 = real_amounts.quantile([0.25, 0.75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr

print(f"Q1 = {q1:,.0f}  Q3 = {q3:,.0f}  IQR = {iqr:,.0f}  Upper fence = {upper:,.0f}")
print("IQR outliers:", clean["is_outlier"].sum())
print(f"Median amount: KES {real_amounts.median():,.0f}")
print(f"Largest amount that is not 250,000: KES "
      f"{clean.loc[~clean['suspected_entry_error'], 'amount'].max():,.0f}")
print()
print("The KES 250,000 transactions")
clean.loc[clean["suspected_entry_error"],
          ["txn_id", "agent_id", "county", "txn_type", "amount", "txn_date"]]

Q1 = 1,077  Q3 = 3,080  IQR = 2,003  Upper fence = 6,084
IQR outliers: 40
Median amount: KES 1,776
Largest amount that is not 250,000: KES 28,847

The KES 250,000 transactions


,txn_id,agent_id,county,txn_type,amount,txn_date
306,TX00307,AG012,Kisumu,Deposit,250000.0,2026-03-02
315,TX00316,AG014,Nairobi,Deposit,250000.0,2026-04-01
424,TX00425,AG005,Mombasa,Withdrawal,250000.0,2026-03-02
463,TX00464,AG009,Nakuru,Withdrawal,250000.0,2026-04-11
594,TX00595,AG016,Kiambu,Deposit,250000.0,2026-03-11


**Outlier findings:** the IQR method flags 40 amounts above KES 6,084. Apart from the five KES 250,000 values, the other 35 are between KES 6,084 and KES 28,847. Mobile money amounts are naturally skewed, with many small transactions and a few large ones, so these look like genuine large transactions and are kept.

**The KES 250,000 values:** five transactions are exactly KES 250,000. That is about 140 times the median transaction (KES 1,776) and almost 9 times the next largest amount. They come from five different agents in five different counties and include both deposits and withdrawals. An identical round number appearing this way points to a system default or data-entry error, not real activity. Left in, they would add KES 1.25M to a clean total of KES 1.44M and almost double it.

**Recommendation:** leave these five out of all totals and send them to the operations team to check against each agent's float records. Correct or remove them only after the real amounts are confirmed.

**Answer:** the merge keeps the row count at 600, so no transactions were lost or duplicated. 7 transactions use agent ID AG099, which is not in the agent register. The raw file had 8, but one of them was a duplicate copy and was removed. A left join keeps these transactions with county set to "Unknown", so their KES 27,156 is not lost. An inner join would have dropped them without warning. The business should ask IT whether AG099 is a test account or a real agent missing from the register.

In [12]:
## 6. Post-Merge Validation

In [13]:
deduped = txns.drop_duplicates(subset="txn_id")

print("Rows before merge:", len(deduped))
print("Rows after merge: ", len(clean))
print("Row count unchanged:", len(deduped) == len(clean))
print("Transactions with unknown agents:", clean["unknown_agent"].sum())
print()
clean.loc[clean["unknown_agent"], ["txn_id", "agent_id", "county", "txn_type", "amount"]]

Rows before merge: 600
Rows after merge:  600
Row count unchanged: True
Transactions with unknown agents: 7



,txn_id,agent_id,county,txn_type,amount
55,TX00056,AG099,Unknown,Deposit,8507.0
68,TX00069,AG099,Unknown,Withdrawal,1877.0
77,TX00078,AG099,Unknown,Deposit,1770.0
237,TX00238,AG099,Unknown,Deposit,4944.0
267,TX00268,AG099,Unknown,Deposit,3110.0
444,TX00445,AG099,Unknown,Deposit,5309.0
557,TX00558,AG099,Unknown,Deposit,1639.0


**Answer:** the merge keeps the row count at 600, so no transactions were lost or duplicated. 7 transactions use agent ID AG099, which is not in the agent register. The raw file had 8, but one of them was a duplicate copy and was removed. A left join keeps these transactions with county set to "Unknown", so their KES 27,156 is not lost. An inner join would have dropped them without warning. The business should ask IT whether AG099 is a test account or a real agent missing from the register.

## 7. Save the Clean Data

In [14]:
clean.to_csv("outputs/agent_transactions_clean.csv", index=False)

check = pd.read_csv("outputs/agent_transactions_clean.csv")
print("Saved outputs/agent_transactions_clean.csv")
print("Rows:", len(check))
print("Duplicate txn_id:", check["txn_id"].duplicated().sum())
print("txn_type values:", sorted(check["txn_type"].unique()))
print("Columns:", list(check.columns))
check.head()

Saved outputs/agent_transactions_clean.csv
Rows: 600
Duplicate txn_id: 0
txn_type values: ['Deposit', 'Withdrawal']
Columns: ['txn_id', 'agent_id', 'txn_type', 'amount', 'txn_date', 'is_outlier', 'suspected_entry_error', 'amount_imputed', 'county', 'unknown_agent']


,txn_id,agent_id,txn_type,amount,txn_date,is_outlier,suspected_entry_error,amount_imputed,county,unknown_agent
0,TX00001,AG009,Deposit,353.0,2026-03-03,False,False,False,Nakuru,False
1,TX00002,AG020,Deposit,1238.0,2026-04-27,False,False,False,Mombasa,False
2,TX00003,AG008,Deposit,9129.0,2026-04-29,True,False,False,Kisumu,False
3,TX00004,AG009,Deposit,652.0,2026-04-24,False,False,False,Nakuru,False
4,TX00005,AG009,Withdrawal,478.0,2026-04-17,False,False,False,Nakuru,False


## 8. Summary Tables

The summaries leave out the five suspected KES 250,000 entry errors. Imputed amounts are included and flagged in the clean file.

In [15]:
analysis = clean[~clean["suspected_entry_error"]]

county_summary = analysis.groupby(["county", "txn_type"]).agg(
    total_value=("amount", "sum"),
    transactions=("txn_id", "count"),
    avg_value=("amount", "mean"),
).round(0)

county_summary

total_value  transactions  avg_value
county  txn_type                                        
Kiambu  Deposit         84737.0            42     2018.0
        Withdrawal      77044.0            27     2853.0
Kisumu  Deposit        186401.0            73     2553.0
        Withdrawal      76368.0            37     2064.0
Mombasa Deposit        148960.0            72     2069.0
        Withdrawal     118592.0            60     1977.0
Nairobi Deposit        293564.0           117     2509.0
        Withdrawal     162010.0            62     2613.0
Nakuru  Deposit        184877.0            60     3081.0
        Withdrawal      78588.0            38     2068.0
Unknown Deposit         25279.0             6     4213.0
        Withdrawal       1877.0             1     1877.0

In [16]:
analysis = analysis.assign(week=analysis["txn_date"].dt.to_period("W").dt.start_time.dt.date)

weekly_pivot = pd.pivot_table(analysis, index="week", columns="county",
                              values="amount", aggfunc="sum", fill_value=0).round(0)
weekly_pivot

county,Kiambu,Kisumu,Mombasa,Nairobi,Nakuru,Unknown
week,,,,,,
2026-02-23,0.0,1893.0,6759.0,7046.0,2737.0,0.0
2026-03-02,18317.0,37933.0,35663.0,68967.0,56574.0,0.0
2026-03-09,17765.0,23186.0,30471.0,52854.0,28730.0,0.0
2026-03-16,22054.0,28112.0,25138.0,43101.0,39795.0,1639.0
2026-03-23,11504.0,27077.0,17050.0,41367.0,18992.0,0.0
2026-03-30,22609.0,41077.0,38344.0,32931.0,18145.0,8507.0
2026-04-06,15030.0,15341.0,47086.0,56780.0,26609.0,9824.0
2026-04-13,12265.0,29403.0,26730.0,56426.0,26146.0,0.0
2026-04-20,24507.0,37439.0,16687.0,65174.0,30862.0,1877.0


## Data Quality Report

**Completeness.** 22 rows in the raw file had no usable amount: 15 blanks and 7 typed as "N/A". After removing duplicates, 20 were left. They were flagged in `amount_imputed` and filled with the median of KES 1,770. No other column had missing values.

**Validity.** Amounts typed as "N/A" could not be read as numbers and were treated as missing. 5 dates used "/" instead of "-" and were converted, so every date is now in ISO format (YYYY-MM-DD). 8 rows in the raw file, 7 after removing duplicates, used agent ID AG099, which does not exist in the agent register. They were kept with county "Unknown".

**Accuracy.** 5 transactions were exactly KES 250,000, about 140 times the median and almost 9 times the next largest amount, across five agents in five counties. They are flagged as `suspected_entry_error` and left out of the summaries until operations confirms them. The IQR method flagged 40 amounts above KES 6,084 in `is_outlier`. The 35 that are not KES 250,000 look like genuine large transactions and are kept.

**Consistency.** txn_type was written five ways: Deposit, deposit, DEPOSIT with a trailing space, Withdrawal and withdrawal. 376 rows were standardised to exactly Deposit or Withdrawal.

**Uniqueness.** 12 transaction IDs appeared twice. Duplicates were found by txn_id and not by the whole row, because one copy of TX00405 had a different agent ID (AG099) and would have been missed. The first copy was kept, leaving 600 unique transactions.

**Timeliness.** The extract covers 1 March to 29 April 2026, about eight weeks. It is suitable for comparing agents and counties in that period, not for current float decisions. The weekly pivot starts on Monday 23 February, so its first week contains only 1 March, and the last week only has data up to 29 April. All agents were onboarded by August 2025, before the first transaction.

**Privacy.** If the extract also held customer phone numbers, the phone numbers would be personal data under the Kenya Data Protection Act 2019, and so would any customer names or ID numbers. Transaction IDs, amounts, dates and agent IDs would also count as personal data once they could be linked to a phone number, because together they show what a named person did and where. Agent IDs may already identify individual agents. To protect this data, I would load only the fields the analysis needs and replace each phone number with a salted hash so customers can still be counted without being named. The salt would stay out of the repository. Any number shown in a report would be masked, for example 0712****78. Results would be reported at county level, with any group of fewer than five customers suppressed. Raw files would be kept out of public GitHub and used only for the purpose the customers were told about.